## Semantic search

In [ ]:
from dotenv import load_dotenv

load_dotenv()

SyntaxError: invalid syntax (1080694654.py, line 1)

In [7]:
import glob
from pprint import pprint 
from langchain_community.document_loaders import PyPDFLoader

pdf_files = glob.glob(r"C:\\Users\\dell\\Documents\\Agentic-AI-Langchain\\Content\\*.pdf")
loader = PyPDFLoader(pdf_files[0])  # Load the first PDF file

data = loader.load()

pprint(data)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2026-06-02T15:41:33+05:30', 'author': '', 'moddate': '2026-06-02T15:41:33+05:30', 'title': 'Microsoft Word - Concept of agent tools and memory in langchain', 'source': 'C:\\\\Users\\\\dell\\\\Documents\\\\Agentic-AI-Langchain\\\\Content\\Concept of agent tools and memory in langchain.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Concept of agent tools and memory in langchain \nAuthor – Sambita Chakraborty \nIn LangChain, Agents act as dynamic reasoning engines that use a Large Language Model (LLM) to decide \na sequence of acƟons. Unlike rigid chains that follow a hardcoded path, an agent evaluates user input, \ndecides which Tools to use, executes them, and passes the results back into its Memory to plan the next \nstep, \nRef: LangChain Deep Dive: Chains, Tools, Agents & Memory | by Amit Kharche | Medium \n        Workﬂows and agents - Docs by LangChain \n \n 1. The 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)

print(len(all_splits))

13


Embedding Models: https://docs.langchain.com/oss/python/integrations/text_embedding

In [9]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [ ]:
ids = vector_store.add_documents(documents=all_splits)

In [ ]:
results = vector_store.similarity_search(
    "How many days of vacation does an employee get in their first year?"
)

print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

## RAG Agent

In [ ]:
from langchain.tools import tool

@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for information"""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [ ]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

llm = ChatOllama(model="llama3.2")

agent = create_agent(
    model=llm,
    tools=[search_handbook],
    system_prompt="You are a helpful agent that can search the employee handbook for information.",
    checkpointer=InMemorySaver()
)

In [ ]:
from langchain.messages import HumanMessage
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke(
    {"messages": [HumanMessage(content="How many days of vacation does an employee get in their first year?")]}, 
    config=config
)

In [ ]:
from pprint import pprint

pprint(response["messages"][-1].content)

('In your first year (0–1 year of service), you get 10 PTO days per year '
 '(about 0.833 days per month). You can carry over up to 5 unused PTO days per '
 'calendar year.')


In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Does Travel policy change if an employee takes more holidays?")]}, 
    config=config
)

In [ ]:

pprint(response["messages"][-1].content)

('No. The Travel & Expense Policy does not change based on how many holidays '
 'you take.\n'
 '\n'
 '- The policy covers reimbursing reasonable and necessary expenses for '
 'approved business travel within established limits.\n'
 '- PTO or holidays are separate from travel reimbursements.\n'
 '- If you’re not on approved business travel (e.g., you’re on vacation), '
 'there are no travel expenses to claim.\n'
 '- If you do have travel, remember to submit receipts within 14 days of '
 'travel.\n'
 '- Extended absences longer than 5 consecutive business days require manager '
 'approval.')


In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is the company's policy on remote work?")]}, 
    config=config
)

pprint(response["messages"][-1].content)

('There isn’t an explicit remote work policy in the handbook sections I can '
 'access. The available policies cover PTO, Travel & Expense, NDA, and '
 'Workplace Conduct, but none specify remote work eligibility, equipment, '
 'security requirements, or approval processes.\n'
 '\n'
 'If you need specifics, please check with HR or your manager for any '
 'department-specific guidance or an addendum. I can also help draft a '
 'proposed remote work policy or search for additional documents if you have '
 'access to more handbook sections.')
